# Sequence Annotation Objects (Biopython) — Full Guided Lesson

## Key concepts (before coding)
- **Seq**: biological sequence object.
- **SeqRecord**: sequence + metadata (`id`, `name`, `description`, `annotations`, `features`, `dbxrefs`, `letter_annotations`).
- **SeqFeature**: biologically meaningful region (gene, CDS, etc.).
- **Location**: feature interval on parent sequence (`start:end`, strand).
- **Position**: single boundary coordinate; exact or fuzzy.
- **Per-letter annotation**: one value per base/residue (e.g., FASTQ quality), must match sequence length.
- **Python indexing**: 0-based, end-exclusive.

## Section plan
1. Setup and files.
2. Build SeqRecord from scratch.
3. Read FASTA and GenBank.
4. Understand features/locations/fuzzy positions.
5. Query coordinates with `in`.
6. Extract feature sequence safely.
7. Compare, format, slice, add, shift, reverse-complement SeqRecord objects.

In [ ]:
# Line 1: Import filesystem path helper.
from pathlib import Path
# Line 2: Import downloader for tutorial files.
from urllib.request import urlretrieve
# Line 3: Import Biopython and SeqIO parser.
import Bio
from Bio import SeqIO

# Line 4: Define data directory inside your project.
data_dir = Path(r"C:/Users/hp/OneDrive/Desktop/projects/bio/data")
# Line 5: Create data directory if missing.
data_dir.mkdir(parents=True, exist_ok=True)

# Line 6: Define required tutorial files and source URLs.
files_to_fetch = {
    "NC_005816.fna": "https://raw.githubusercontent.com/biopython/biopython/master/Tests/GenBank/NC_005816.fna",
    "NC_005816.gb": "https://raw.githubusercontent.com/biopython/biopython/master/Tests/GenBank/NC_005816.gb",
    "example.fastq": "https://raw.githubusercontent.com/biopython/biopython/master/Tests/Quality/example.fastq",
}

# Line 7: Download files only when absent.
for name, url in files_to_fetch.items():
    target = data_dir / name
    if not target.exists():
        try:
            urlretrieve(url, target)
            print("Downloaded:", name)
        except Exception as exc:
            print("Download failed:", name, "->", exc)

# Line 8: Print environment and file readiness.
print("Biopython version:", Bio.__version__)
for name in files_to_fetch:
    print("Ready path:", data_dir / name)

## 1) SeqRecord from scratch

A minimal SeqRecord needs only a `Seq`, but practical usage sets:
- id/name/description
- annotations
- per-letter annotations
- dbxrefs
- features

In [ ]:
# Line 1: Import sequence and record classes.
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord
# Line 2: Import feature classes.
from Bio.SeqFeature import SeqFeature, SimpleLocation

# Line 3: Create a DNA sequence.
simple_seq = Seq("GATC")
# Line 4: Wrap it into a SeqRecord.
simple_seq_r = SeqRecord(simple_seq)

# Line 5: Show default identity placeholders.
print("Default id:", simple_seq_r.id)
print("Default name:", simple_seq_r.name)
print("Default description:", simple_seq_r.description)

# Line 6: Set identity metadata.
simple_seq_r.id = "AC12345"
simple_seq_r.name = "DemoClone"
simple_seq_r.description = "Made up sequence I wish I could write a paper about"

# Line 7: Add general annotation dictionary entries.
simple_seq_r.annotations["evidence"] = "None. I just made it up."

# Line 8: Add per-letter quality values (length must equal sequence length).
simple_seq_r.letter_annotations["phred_quality"] = [40, 40, 38, 30]

# Line 9: Add an external database cross-reference.
simple_seq_r.dbxrefs.append("Project:Demo123")

# Line 10: Add a feature object (gene on interval [1:3], + strand).
demo_feature = SeqFeature(
    SimpleLocation(1, 3, strand=1),
    type="gene",
    qualifiers={"gene": ["demoA"]},
)
simple_seq_r.features.append(demo_feature)

# Line 11: Display the complete object state.
print(simple_seq_r)
print("seq:", simple_seq_r.seq)
print("annotations:", simple_seq_r.annotations)
print("letter_annotations:", simple_seq_r.letter_annotations)
print("dbxrefs:", simple_seq_r.dbxrefs)
print("features count:", len(simple_seq_r.features))

## 2) Read a FASTA record

FASTA usually gives:
- id and name from first token after `>`
- description from full title line
- little/no structured annotation

In [ ]:
# Line 1: Read one FASTA record.
record_fasta = SeqIO.read(data_dir / "NC_005816.fna", "fasta")

# Line 2: Print compact summary.
print(record_fasta)

# Line 3: Inspect identity fields.
print("id:", record_fasta.id)
print("name:", record_fasta.name)
print("description:", record_fasta.description)

# Line 4: FASTA generally lacks rich metadata.
print("dbxrefs:", record_fasta.dbxrefs)
print("annotations:", record_fasta.annotations)
print("letter_annotations:", record_fasta.letter_annotations)
print("features:", record_fasta.features)

## 3) Read a GenBank record

GenBank is richer:
- annotations dictionary populated
- feature list populated
- dbxrefs may be present

In [ ]:
# Line 1: Read one GenBank record.
record_gb = SeqIO.read(data_dir / "NC_005816.gb", "genbank")

# Line 2: Print summary.
print(record_gb)

# Line 3: Show identity fields.
print("id:", record_gb.id)
print("name:", record_gb.name)
print("description:", record_gb.description)

# Line 4: Show rich metadata.
print("len(annotations):", len(record_gb.annotations))
print("source annotation:", record_gb.annotations.get("source"))
print("dbxrefs:", record_gb.dbxrefs)
print("len(features):", len(record_gb.features))
print("letter_annotations:", record_gb.letter_annotations)

## 4) Inspect feature structure (`type`, `location`, `strand`, `qualifiers`)

In [ ]:
# Line 1: Pick tutorial feature indices when available.
indices = [20, 21] if len(record_gb.features) > 21 else [0]

# Line 2: Inspect each selected feature.
for idx in indices:
    feature = record_gb.features[idx]
    print("\nFeature index:", idx)
    print("type:", feature.type)
    print("location:", feature.location)
    print("strand:", feature.location.strand)
    print("qualifier keys:", sorted(feature.qualifiers.keys()))
    print("db_xref:", feature.qualifiers.get("db_xref"))

## 5) Positions and fuzzy locations

Key idea:
- position = one boundary
- location = interval between boundaries

Fuzzy examples include `AfterPosition`, `BeforePosition`, and `BetweenPosition`.

In [ ]:
# Line 1: Import SeqFeature namespace for position/location classes.
from Bio import SeqFeature as SF

# Line 2: Create fuzzy start (>5).
start_pos = SF.AfterPosition(5)

# Line 3: Create fuzzy end with compatibility fallback.
if hasattr(SF, "BetweenPosition"):
    end_pos = SF.BetweenPosition(9, left=8, right=9)
else:
    end_pos = SF.WithinPosition(9, left=8, right=9)

# Line 4: Build fuzzy location.
my_location = SF.SimpleLocation(start_pos, end_pos)

# Line 5: Print fuzzy location and integer conversions.
print("my_location:", my_location)
print("start object:", my_location.start, "| int:", int(my_location.start))
print("end object:", my_location.end, "| int:", int(my_location.end))

# Line 6: Build exact location from integers.
exact_location = SF.SimpleLocation(5, 9)
print("exact_location:", exact_location)
print("exact start type:", type(exact_location.start).__name__)
print("exact end type:", type(exact_location.end).__name__)

## 6) Coordinate membership testing with `in`

Use case: “Which features contain this SNP coordinate?”

In [ ]:
# Line 1: Define a SNP coordinate in Python indexing.
my_snp = 4350

# Line 2: Iterate over features and test membership.
hits = []
for feature in record_gb.features:
    if my_snp in feature:
        hits.append((feature.type, feature.qualifiers.get("db_xref")))

# Line 3: Print all overlapping features.
print("SNP coordinate:", my_snp)
for feature_type, refs in hits:
    print(feature_type, refs)

## 7) Extract sequence described by a feature

Preferred method: `feature.extract(parent_seq)` because it handles strand and complex locations.

In [ ]:
# Line 1: Import compact classes for toy demonstration.
from Bio.Seq import Seq
from Bio.SeqFeature import SeqFeature, SimpleLocation

# Line 2: Define a short parent sequence.
seq = Seq("ACCGAGACGGCAAAGGCTAGCATAGGTATGAGACTTCCTTCCTGCCAGTGCTGAGGAACTGGGAGCCTAC")

# Line 3: Define reverse-strand feature over [5:18].
feature = SeqFeature(SimpleLocation(5, 18, strand=-1), type="gene")

# Line 4: Manual extraction (slice then reverse complement).
manual = seq[feature.location.start : feature.location.end].reverse_complement()

# Line 5: Built-in extraction.
auto = feature.extract(seq)

# Line 6: Validate equivalence and lengths.
print("manual:", manual)
print("auto  :", auto)
print("same? :", manual == auto)
print("len(auto):", len(auto))
print("len(feature):", len(feature))
print("len(feature.location):", len(feature.location))

## 8) SeqRecord comparison

Direct `record1 == record2` is deliberately not implemented.
Compare explicit attributes (`id`, `seq`, etc.) instead.

In [ ]:
# Line 1: Import minimal classes.
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

# Line 2: Create two records with same visible content.
record1 = SeqRecord(Seq("ACGT"), id="test")
record2 = SeqRecord(Seq("ACGT"), id="test")

# Line 3: Show modern comparison behavior.
try:
    print(record1 == record2)
except NotImplementedError as exc:
    print("Expected NotImplementedError:", exc)

# Line 4: Compare explicit attributes safely.
print("id equal:", record1.id == record2.id)
print("seq equal:", record1.seq == record2.seq)

## 9) `format()` method

`record.format("fasta")` returns a text string in the chosen output format.

In [ ]:
# Line 1: Import classes for formatting example.
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

# Line 2: Build a protein record.
record_fmt = SeqRecord(
    Seq(
        "MMYQQGCFAGGTVLRLAKDLAENNRGARVLVVCSEITAVTFRGPSETHLDSMVGQALFGD"
        "GAGAVIVGSDPDLSVERPLYELVWTGATLLPDSEGAIDGHLREVGLTFHLLKDVPGLISK"
        "NIEKSLKEAFTPLGISDWNSTFWIAHPGGPAILDQVEAKLGLKEEKMRATREVLSEYGNM"
        "SSAC"
    ),
    id="gi|14150838|gb|AAK54648.1|AF376133_1",
    description="chalcone synthase [Cucumis sativus]",
)

# Line 3: Convert record to FASTA text.
fasta_text = record_fmt.format("fasta")

# Line 4: Print result.
print(fasta_text)

## 10) Slice a SeqRecord

When slicing:
- sequence is sliced
- per-letter annotations are sliced
- fully contained features are kept and re-based
- most global annotations/dbxrefs are dropped for safety

In [ ]:
# Line 1: Slice a region that includes pim gene/CDS.
sub_record = record_gb[4300:4800]

# Line 2: Print size and feature count.
print("len(sub_record):", len(sub_record))
print("len(sub_record.features):", len(sub_record.features))

# Line 3: Show adjusted feature locations.
for f in sub_record.features:
    print(f.type, f.location)

# Line 4: Show preserved vs dropped metadata.
print("annotations:", sub_record.annotations)
print("dbxrefs:", sub_record.dbxrefs)

# Line 5: Fix context-specific metadata for partial fragment.
sub_record.annotations["topology"] = "linear"
sub_record.description = "Yersinia pestis biovar Microtus str. 91001 plasmid pPCP1, partial"

# Line 6: Preview GenBank text.
print(sub_record.format("genbank")[:240] + "...")

## 11) Add SeqRecords (FASTQ edit pattern)

`edited = left + right` keeps compatible per-letter annotations aligned.

In [ ]:
# Line 1: Read first FASTQ record.
record_fastq = next(SeqIO.parse(data_dir / "example.fastq", "fastq"))

# Line 2: Inspect original sequence and qualities.
print("Original length:", len(record_fastq))
print("Original seq:", record_fastq.seq)
print("Original qualities:", record_fastq.letter_annotations["phred_quality"])

# Line 3: Remove one suspected extra base by slicing around index 20.
left = record_fastq[:20]
right = record_fastq[21:]

# Line 4: Concatenate slices to create edited record.
edited = left + right

# Line 5: Validate output sequence and qualities.
print("Edited length:", len(edited))
print("Edited seq:", edited.seq)
print("Edited qualities:", edited.letter_annotations["phred_quality"])

## 12) Shift circular origin by record addition

Pattern:
`shifted = record[cut:] + record[:cut]`

In [ ]:
# Line 1: Shift origin at coordinate 2000.
shifted = record_gb[2000:] + record_gb[:2000]

# Line 2: Compare size/features before and after.
print("Original length:", len(record_gb), "| Shifted length:", len(shifted))
print("Original features:", len(record_gb.features), "| Shifted features:", len(shifted.features))

# Line 3: Show cautious metadata loss after slicing+adding.
print("shifted.dbxrefs before:", shifted.dbxrefs)
print("shifted.annotations keys before:", list(shifted.annotations.keys()))

# Line 4: Restore metadata explicitly if biologically valid.
shifted.dbxrefs = record_gb.dbxrefs[:]
shifted.annotations = record_gb.annotations.copy()

# Line 5: Confirm restored metadata.
print("shifted.dbxrefs after:", shifted.dbxrefs)
print("shifted.annotations keys after:", list(shifted.annotations.keys()))

## 13) Reverse-complement a SeqRecord

`reverse_complement()`:
- reverse-complements sequence
- remaps feature locations/strands
- reverses per-letter annotations
- drops many metadata fields by default unless requested

In [ ]:
# Line 1: Alias original record.
rec = record_gb

# Line 2: Reverse-complement and set new id.
rc = rec.reverse_complement(id="TESTING")

# Line 3: Compare summary statistics.
print("Original:", rec.id, len(rec), len(rec.features), len(rec.dbxrefs), len(rec.annotations))
print("RC      :", rc.id, len(rc), len(rc.features), len(rc.dbxrefs), len(rc.annotations))

## Final recap

You now have a full, sectioned, teaching-first notebook with:
- definitions before implementation
- line-commented code
- practical examples throughout